### **Student Information**
GitHub ID: pablolescabo

Kaggle name: Paul Tapponnier (@paultapponnier)

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

***

# **Project Report**

---

## 1. Model Development

**Task.** 4,800 short posts (median 13 words), 3,360 labelled for training and 1,440 to predict, over 6 emotions: joy 875, anger 674, sadness 628, disgust 454, surprise 441, fear 288. The metric is macro-F1, so every class counts equally and the rare class (*fear*) matters as much as *joy*.

**Results summary**

| Model | 5-fold CV macro-F1 | Public LB |
| --- | --- | --- |
| TF-IDF baseline (word 1-2 + char 2-5, Linear SVM) | 0.520 | 0.5378 |
| Tuned TF-IDF (word 1-3 + cased char 2-5, Linear SVM C=0.1) | 0.529 | not submitted |
| RoBERTa-large, j-hartmann emotion checkpoint (+ bias tuning) | 0.636 | 0.6347 |
| RoBERTa-large, cardiffnlp Twitter emotion checkpoint (+ bias tuning) | 0.628 | not submitted |
| **Ensemble of both RoBERTa models (+ bias tuning)** | **0.650** | 0.6346 |

The public leaderboard uses only ~30% of the test set (~430 posts), so differences of a few thousandths are noise. The two transformer submissions are tied on the public leaderboard (0.6347 vs 0.6346), but cross-validation over 3,360 posts gives the ensemble a clear +0.014 lead. All model choices were therefore made on the CV score. The ensemble and the single j-hartmann model are the two submissions selected for the final (private) ranking.

### 1.1 Preprocessing Steps

* **Loading and merging:** `final_posts.json` (text), `emotion.csv` (labels) and `data_identification.csv` (train/test split) are joined on `post_id`, then split into train and test sets.
* **Light text cleaning:**
    * curly quotes and apostrophes are normalised (`don’t` → `don't`), so both spellings map to the same token;
    * URLs are replaced by `http` and user mentions by `@user`, since their exact content carries no emotion;
    * repeated whitespace is collapsed.
* **What was deliberately kept:** casing, punctuation and emojis. Cues such as "I HATE THIS" or "WHY?!" carry emotional intensity. The experiment in 1.2 confirms that keeping capital letters helps, and the transformer tokenizers are case-sensitive.
* **Labels** are encoded as integers 0–5.
* **Validation split:** a single stratified 5-fold split (seed 42) is shared by every model and every experiment, so all scores are directly comparable. It also allows the out-of-fold predictions of separately trained models to be combined.
* **Submission format:** predictions are written in the order of `sampleSubmission.csv` with the header `id,label` (the sample file uses `post_id`, but Kaggle's checker expects `id`).

### 1.2 Feature Engineering Steps

**Sparse features for the baseline.** Every experiment below uses the same 5 folds. The vectorizers are fitted inside each training fold, so there is no leakage.

**(a) Representation: BoW vs TF-IDF** (word 1-3 + char 2-5, macro-F1)

| Representation | Word only + SVM | Word + char + SVM | Word + char + LR | Word + char + NB |
| --- | --- | --- | --- | --- |
| BoW counts | 0.488 | 0.467 | 0.471 | 0.458 |
| BoW binary | 0.487 | 0.464 | 0.478 | 0.459 |
| TF, normalised, no IDF | 0.483 | 0.514 | 0.506 | 0.266 |
| **TF-IDF** | 0.493 | **0.529** | 0.522 | 0.340 |

On words alone, BoW and TF-IDF are almost equal: in 13-word posts most words appear once, so counting and weighting barely differ. Once character n-grams are added, BoW collapses (0.467) while TF-IDF improves (0.529). Two mechanisms explain this:
* **L2 normalisation** stops long posts and the many character n-grams from dominating. It accounts for most of the gain (0.467 → 0.514).
* **IDF** down-weights uninformative pieces such as `th` or `e ` that appear everywhere (+0.015).

Naive Bayes behaves the other way round: it is designed for counts and works best with BoW.

**(b) N-gram size** (Linear SVM, C=0.3, macro-F1)

| Word n-grams ↓ / char n-grams → | none | (2,5) | (1,6) | (3,6) |
| --- | --- | --- | --- | --- |
| (1,1) | 0.494 | 0.511 | 0.513 | 0.507 |
| (1,2) | 0.497 | 0.520 | 0.519 | 0.520 |
| (1,3) | 0.493 | 0.522 | 0.521 | 0.519 |

Longer word n-grams bring almost nothing, and without character features word trigrams even hurt. The posts are too short for longer sequences to repeat:

| | Distinct in train | Seen in a single post |
| --- | --- | --- |
| words | 7,545 | 60% |
| word bigrams | 29,549 | 85% |
| word trigrams | 37,650 | 96% |
| word 4-grams | 36,335 | 99% |

A pattern seen once cannot generalise. Character n-grams give the largest gain (+0.02) because they share evidence across spelling variants ("scaaared"), inflections ("disgust/disgusted/disgusting") and words unseen in training.

**(c) Longer character n-grams and casing** (word 1-3, Linear SVM C=0.1)

| Char n-grams | Lowercase | Macro-F1 |
| --- | --- | --- |
| (2,5) | yes | 0.523 |
| (2,6) | yes | 0.524 |
| (2,7) | yes | 0.521 |
| (2,8) | yes | 0.520 |
| (4,8) | yes | 0.518 |
| **(2,5)** | **no** | **0.529** |

Character n-grams longer than 6 approach whole words and run into the same sparsity problem. Repetitions such as "soooo" or "!!!!" are already captured by short pieces (`ooo`, `!!!`). The best result comes from **keeping capital letters**: shouting ("I HATE THIS") is a signal of intensity that lowercasing erased.

**Conclusion of the sparse-feature study:** about 60 configurations stay within 0.52–0.53. The ceiling comes from the bag-of-words representation itself, which ignores word order, context and irony, and this motivated the transformer.

**Learned features for the transformer.** Texts are tokenized with the case-sensitive RoBERTa BPE tokenizer (truncated to 128 tokens, which covers almost every post) and dynamically padded per batch. No hand-crafted features are used: contextual representations are learned by the pretrained encoder and adapted during fine-tuning.

### 1.3 Explanation of Your Model

**Baseline.** `LinearSVC(C=0.1, class_weight='balanced')` on TF-IDF word 1-3 grams (English stop words removed) plus cased character 2-5 grams. Among the classifiers compared, the SVM (best C among 0.05–1) beat logistic regression (0.519) and Complement NB (0.513). Balanced class weights counter the imbalance (joy is 3× more frequent than fear).

**How the transformer works.** RoBERTa-large splits a post into sub-word tokens and turns each one into a 1,024-dimensional vector. 24 self-attention layers then let every token update its vector using all the other tokens. In "I'm **not** happy", for example, the representation of "happy" absorbs the negation, which a bag of words cannot do. A summary vector (the `<s>` token) is fed to a classification head. The model's knowledge is stacked in three stages: pretraining on ~160 GB of English text, specialisation on emotion datasets, and finally our fine-tuning on 3,360 posts.

**Two emotion-pretrained checkpoints**, both RoBERTa-large:
* `j-hartmann/emotion-english-roberta-large`: trained on several English emotion datasets for Ekman's six emotions plus *neutral*, exactly our label set.
* `cardiffnlp/twitter-roberta-large-emotion-latest`: RoBERTa-large further pretrained on tweets, then trained on 11 Twitter emotions (including our six). It was chosen because our posts are short and informal.

**Fine-tuning setup (identical for both):**
* **Head reuse:** both checkpoints already have outputs for our six labels. Instead of a new random layer, the existing head is kept and our labels are mapped to it; the other outputs (*neutral*, *love*, *optimism*…) are ignored.
* **Optimisation:** AdamW, learning rate 1e-5, weight decay 0.01, batch size 16, 4 epochs, linear schedule with 10% warm-up, gradient clipping at 1.0, label smoothing 0.05, mixed precision (fp16). Each model takes ~22 minutes on a Kaggle T4 GPU.
* **5-fold cross-validation:** one model per fold, keeping the epoch with the best validation macro-F1 (a form of early stopping). The validation predictions form out-of-fold (OOF) predictions over the whole training set, and the test probabilities of the 5 fold models are averaged.
* **Ensemble:** the class probabilities of the two checkpoints are averaged. This is possible because both use exactly the same folds. In practice the j-hartmann model was trained in a first run and its saved OOF/test predictions were reloaded in the second run, so it did not have to be retrained.
* **Per-class bias tuning:** one additive offset per class is searched on the OOF probabilities to maximise macro-F1, then applied to the test probabilities.

**Best validation macro-F1 per fold**

| Fold | j-hartmann | cardiffnlp |
| --- | --- | --- |
| 0 | **0.637** | 0.588 |
| 1 | 0.587 | **0.608** |
| 2 | **0.655** | 0.627 |
| 3 | **0.665** | 0.654 |
| 4 | **0.620** | 0.605 |
| OOF (+ bias tuning) | **0.636** | 0.628 |

The best epoch was usually the 2nd or 3rd: both models adapt quickly because they already knew the emotions. **cardiffnlp alone is weaker** (0.628 vs 0.636). Its Twitter pretraining did not pay off on its own, possibly because our data also contains news-style sentences ("The pottery around Brasov is in danger"). It also over-predicts *anger* and under-predicts *disgust*, which is why bias tuning helps it much more (+0.011 vs +0.002), probably a legacy of its multi-label training. **Combined, the two models gain +0.014** (0.650): pretrained on different data, they do not fail on the same posts, and averaging corrects part of each model's errors.

**Per-class OOF results of the final ensemble**

| Class | Precision | Recall | F1 |
| --- | --- | --- | --- |
| anger | 0.630 | 0.644 | 0.637 |
| disgust | 0.510 | 0.469 | 0.489 |
| fear | 0.684 | 0.670 | 0.677 |
| sadness | 0.674 | 0.689 | 0.682 |
| surprise | 0.638 | 0.617 | 0.627 |
| joy | 0.774 | 0.799 | 0.786 |
| **macro avg** | 0.652 | 0.648 | **0.650** |

*Disgust* remains the hardest class, most likely confused with *anger* (both express negative judgement). *Fear* does well despite being the rarest class.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude (Anthropic) as an assistant during the competition. The prompts below are condensed and translated from French, in chronological order. For each one: what I asked, what the AI suggested, and what I decided.

**1. Understanding the task before modelling**
> *"Here are the competition instructions and the four data files. Where should we start?"*

The AI first inspected the data instead of proposing a model straight away. It found strong class imbalance (joy 875 vs fear 288) and very short texts (median 13 words), and pointed out that the metric is macro-F1. It proposed a TF-IDF baseline followed by a fine-tuned transformer. It also noticed that the six labels are exactly Ekman's basic emotions, so checkpoints already trained on them exist (`j-hartmann/emotion-english-*`).
**Decision: adopted.** The imbalance and the metric made me use balanced class weights and stratified folds everywhere. The Ekman remark led to my first transformer.

**2. Building a measurable progression**
> *"Let's use a Kaggle notebook (free GPU) and do TF-IDF first, then a transformer: we have several submissions per day, and each step should show the optimisation. The current public best is 0.6281."*

This was my decision, with three goals: get a reference score, check the submission format early, and justify the heavier model with evidence. The AI compared five TF-IDF variants on the same folds.
**Decision: adopted.** The first submission scored CV 0.520 and public 0.5378.

**3. Understanding every step**
> *"Explain each step of the baseline in detail: TF-IDF, n-grams, the classifiers, cross-validation and macro-F1."*

I used the explanation to check the methodology rather than take the code as given. In particular, the vectorizers are fitted inside each fold, so there is no leakage, and scikit-learn's stop-word list also removes "not".

**4. First transformer: fine-tuning design suggested by the AI**

The AI suggested:
* reusing the pretrained emotion head instead of a new random layer;
* 5-fold training keeping the best epoch per fold, with test probabilities averaged over the folds;
* per-class bias tuning on the OOF predictions.

**Decision: adopted.** j-hartmann reached CV 0.636 and public 0.6347, 1st on the public leaderboard at that time. Bias tuning only added +0.002, which I read as a sign that the model was already balanced.

**5. Reading the leaderboard critically**
> *"I scored 0.5378 on the public leaderboard instead of the 0.52 from cross-validation. Why?"*

The AI explained that the public leaderboard covers only ~430 posts, so ±0.02–0.03 is expected noise, while CV on 3,360 posts is the more reliable estimate.
**Decision: adopted.** I compared models only on CV from then on.

**6. Strengthening the optimisation with real experiments**
> *"How can I make the optimisation of each submission stronger?"*

The AI recommended running actual experiments with measurable results: a TF-IDF hyperparameter search, then new transformer variants.
**Decision: adopted.** A search over ~60 TF-IDF configurations (stop words, word and char n-gram ranges, classifier, C) moved the baseline only from 0.520 to 0.522. The bag-of-words representation itself was the ceiling. I did not submit this marginal improvement.

**7. Questioning the baseline myself**

These were my own questions. Each one was answered with a measured experiment rather than a general explanation, and they make up the feature study in section 1.2.
* *"Could other n-gram sizes give better results, or only a few more details?"* Longer word n-grams barely help, because 96% of word trigrams occur in a single post. I then concluded that character n-grams work better because they handle misspellings, and the AI added that they also share evidence across inflections and unseen words.
* *"So we could make the character n-grams longer to catch some extremes."* Longer character n-grams did not help (peak at 6, then decline). But the idea of capturing intensity led to testing **case-sensitive** character n-grams, which gave the best TF-IDF score (0.529): capital letters signal shouting.
* *"Why TF-IDF rather than BoW? Is BoW too basic?"* BoW equals TF-IDF on words alone but collapses with character n-grams (0.467 vs 0.529), because it lacks normalisation and IDF weighting. Naive Bayes behaves the opposite way.

**Decision:** all three results were kept as evidence for the final feature choices (TF-IDF, word 1-3, cased char 2-5).

**8. Alternatives and understanding the transformer**
> *"Weren't there simpler alternatives to the transformer? And what does the transformer actually do?"*

The AI listed the intermediate options with their trade-offs: an emotion lexicon (NRC), averaged word embeddings, an LSTM/CNN trained from scratch, frozen sentence embeddings with logistic regression, zero-shot use of the pretrained model, and smaller transformers. It then explained tokenization, self-attention and the three stacked training stages.
**Decision:** I used the explanation for section 1.3. I did not run the intermediate alternatives, to spend the GPU time on the more promising transformer variants, so they remain unmeasured.

**9. Choosing a better pretrained model for short, informal texts**
> *"Are there models even better trained for this kind of data, short and very informal?"*

The AI proposed `cardiffnlp/twitter-roberta-large-emotion-latest` (Twitter-pretrained, emotion head including our six labels), `vinai/bertweet-large`, `SamLowe/roberta-base-go_emotions` (Reddit comments) and `microsoft/deberta-v3-large` for ensembling. It recommended an ensemble as the most reliable gain.
**Decision:** I chose **cardiffnlp**, because it matches our case (informal social-media text and emotion labels). The AI then suggested reusing the j-hartmann predictions saved by the first run, thanks to identical folds, instead of retraining it.
**Rejected:** DeBERTa-v3-large. A notebook was prepared but not run: I preferred the domain-matched model, and two large models fitted my time budget better than three.

**10. Interpreting the comparison**
> *"Why does cardiffnlp do better on the fold where j-hartmann is weak?"*

The AI answered that a single fold out of five is weak evidence and most likely noise. What matters for the ensemble is that the two models do not fail on the same posts, and the +0.014 ensemble gain shows this. It proposed an error analysis on the OOF files to test the "more informal posts" hypothesis.
**Decision:** I reported the fold table but did not claim a cause, since the error analysis was not carried out.

**11. Choosing the final submissions**
> *"The ensemble scored 0.6346, less than my previous submission (0.6347)."*

The AI pointed out that a 0.0001 difference on ~430 posts is a tie, whereas CV on 3,360 posts favours the ensemble by +0.014. Since Kaggle allows two final submissions and keeps the better one, it suggested selecting both.
**Decision: adopted.** Final submissions: the ensemble (CV 0.650) and j-hartmann alone (CV 0.636).

**12. Debugging, and verifying the AI's assumptions**
* *"ID column id not found"*: the AI had followed `sampleSubmission.csv`, which uses `post_id`, while the competition overview specifies `id`. I had it fix the output to `id,label`. Lesson: check AI-generated code against the official specification, not only against the example files.
* `IndexError` when loading data: the imported notebook had no input attached. The fix was to add the competition dataset as notebook input.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
# Run on Kaggle (GPU T4, internet on). Inputs: competition data (+ optionally earlier runs' .npy predictions)
import os, glob, re, random, time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

SEED = 42
LABELS = ['anger', 'disgust', 'fear', 'sadness', 'surprise', 'joy']

def find(name):
    # Kaggle input folder first, then local folders
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True)
            or glob.glob(f'**/{name}', recursive=True)
            or glob.glob(f'../**/{name}', recursive=True))
    return hits[0]

posts  = pd.read_json(find('final_posts.json'))
emo    = pd.read_csv(find('emotion.csv'))
ident  = pd.read_csv(find('data_identification.csv'))
sample = pd.read_csv(find('sampleSubmission.csv'))

df = posts.merge(ident, on='post_id').merge(emo, on='post_id', how='left')
train = df[df.set == 'train'].reset_index(drop=True)
test  = df[df.set == 'test'].reset_index(drop=True)
print(train.shape, test.shape)
print(train.label.value_counts())
print('Words per post:', df.text.str.split().str.len().describe()[['50%', 'mean', 'max']].to_dict())

In [ ]:
# Light cleaning: keep casing, punctuation and emojis
def clean(t):
    t = str(t).replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"')
    t = re.sub(r'https?://\S+|www\.\S+', 'http', t)
    t = re.sub(r'@\w+', '@user', t)
    return re.sub(r'\s+', ' ', t).strip()

train['clean'] = train.text.map(clean)
test['clean']  = test.text.map(clean)

lab2id = {l: i for i, l in enumerate(LABELS)}
y = train.label.map(lab2id).values
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y))  # shared by all models

def save_submission(pred, path):
    sub = pd.DataFrame({'post_id': test.post_id, 'label': [LABELS[i] for i in pred]})
    sub = sub.set_index('post_id').loc[sample.post_id].reset_index()  # same order as sample
    sub = sub.rename(columns={'post_id': 'id'})  # Kaggle expects 'id'
    sub.to_csv(path, index=False)
    print(path, sub.shape, sub.label.value_counts().to_dict())

## 2. Feature Engineering Steps

### 2.1 Sparse text features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB

def word(V=TfidfVectorizer, ng=(1, 3), stop='english', **k):
    return V(ngram_range=ng, stop_words=stop, **k)

def char(V=TfidfVectorizer, ng=(2, 5), lowercase=False, **k):
    return V(analyzer='char_wb', ngram_range=ng, min_df=2, lowercase=lowercase, **k)

def svm(C=0.1):
    return LinearSVC(C=C, class_weight='balanced', max_iter=10000)

def cv(model):
    # 5-fold macro-F1; vectorizers are refitted inside each fold (no leakage)
    return cross_val_score(model, train.clean, y, cv=folds, scoring='f1_macro').mean()

### 2.2 Representation: BoW vs TF-IDF

In [ ]:
reps = {
    'BoW counts':  dict(V=CountVectorizer),
    'BoW binary':  dict(V=CountVectorizer, binary=True),
    'TF (no IDF)': dict(V=TfidfVectorizer, use_idf=False, sublinear_tf=True),
    'TF-IDF':      dict(V=TfidfVectorizer, sublinear_tf=True),
}
rows = []
for name, kw in reps.items():
    feats = lambda: make_union(word(**kw), char(**kw))
    rows.append({'representation': name,
                 'word only + SVM': cv(make_pipeline(word(**kw), svm())),
                 'word+char + SVM': cv(make_pipeline(feats(), svm())),
                 'word+char + LR':  cv(make_pipeline(feats(), LogisticRegression(C=1, max_iter=5000, class_weight='balanced'))),
                 'word+char + NB':  cv(make_pipeline(feats(), MultinomialNB(alpha=0.5)))})
pd.DataFrame(rows).round(4)

### 2.3 N-gram sizes and sparsity

In [ ]:
# Share of word n-grams that occur in a single training post
for n in [1, 2, 3, 4]:
    dfreq = (CountVectorizer(ngram_range=(n, n)).fit_transform(train.text) > 0).sum(0).A1
    print(f'{n}-grams: {len(dfreq):6d} distinct, {100 * (dfreq == 1).mean():.0f}% in one post only')

tf = dict(sublinear_tf=True)
grid = {}
for wng in [(1, 1), (1, 2), (1, 3)]:
    grid[(str(wng), 'none')] = cv(make_pipeline(word(ng=wng, **tf), svm(0.3)))
    for cng in [(2, 5), (1, 6), (3, 6)]:
        grid[(str(wng), str(cng))] = cv(make_pipeline(make_union(word(ng=wng, **tf), char(ng=cng, lowercase=True, **tf)), svm(0.3)))
pd.Series(grid).unstack().round(4)

### 2.4 Longer character n-grams and casing

In [ ]:
rows = []
for ng, low in [((2, 5), True), ((2, 6), True), ((2, 7), True), ((2, 8), True), ((4, 8), True), ((2, 5), False)]:
    rows.append({'char n-grams': str(ng), 'lowercase': low,
                 'F1': cv(make_pipeline(make_union(word(**tf), char(ng=ng, lowercase=low, **tf)), svm(0.1)))})
pd.DataFrame(rows).round(4)

### 2.5 Tokenization for the transformer

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup

MAX_LEN = 128

def encode(tok, texts, labels=None):
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
    items = [{k: enc[k][i] for k in enc} for i in range(len(texts))]
    if labels is not None:
        for it, l in zip(items, labels):
            it['labels'] = int(l)
    return items

## 3. Model Implementation Steps

### 3.1 Baseline: classifier choice and final TF-IDF model

In [ ]:
best_feats = lambda: make_union(word(**tf), char(**tf))  # word 1-3 (no stop words) + cased char 2-5
rows = [{'classifier': 'ComplementNB', 'C': '-', 'F1': cv(make_pipeline(best_feats(), ComplementNB(alpha=0.3)))}]
for C in [0.05, 0.1, 0.3, 1.0]:
    rows.append({'classifier': 'LinearSVC', 'C': C, 'F1': cv(make_pipeline(best_feats(), svm(C)))})
for C in [2, 5, 20]:
    rows.append({'classifier': 'LogisticRegression', 'C': C,
                 'F1': cv(make_pipeline(best_feats(), LogisticRegression(C=C, max_iter=5000, class_weight='balanced')))})
pd.DataFrame(rows).round(4)

In [ ]:
# Refit the tuned baseline on all training data
tfidf_model = make_pipeline(best_feats(), svm(0.1)).fit(train.clean, y)
save_submission(tfidf_model.predict(test.clean), 'submission_tfidf.csv')

### 3.2 Fine-tuned RoBERTa-large models, 5-fold

In [ ]:
# Both checkpoints already have emotion heads covering our six labels.
# They were trained in two Kaggle runs with identical folds and settings; listing both here reproduces everything in one run.
MODELS = {
    'j-hartmann/emotion-english-roberta-large':        dict(lr=1e-5, epochs=4),
    'cardiffnlp/twitter-roberta-large-emotion-latest': dict(lr=1e-5, epochs=4),
}
BATCH_SIZE = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def build_model(name):
    m = AutoModelForSequenceClassification.from_pretrained(name)
    l2i = {k.lower(): v for k, v in (m.config.label2id or {}).items()}
    if all(l in l2i for l in LABELS):
        idx = [l2i[l] for l in LABELS]  # reuse pretrained head, other outputs ignored
    else:
        m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(LABELS), ignore_mismatched_sizes=True)
        idx = list(range(len(LABELS)))
    return m, idx

@torch.no_grad()
def predict(model, loader, idx):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
            logits = model(**b).logits.float()
        out.append(torch.softmax(logits[:, idx], -1).cpu().numpy())
    return np.concatenate(out)

def run_model(name, LR, EPOCHS):
    tok = AutoTokenizer.from_pretrained(name)
    coll = DataCollatorWithPadding(tok)
    test_dl = DataLoader(encode(tok, test.clean), batch_size=64, collate_fn=coll)
    oof = np.zeros((len(train), len(LABELS)))
    test_prob = np.zeros((len(test), len(LABELS)))
    for f, (tr_i, va_i) in enumerate(folds):
        seed_all(SEED + f)
        model, idx = build_model(name)
        model.to(device)
        idx_t = torch.tensor(idx, device=device)
        tr_dl = DataLoader(encode(tok, train.clean[tr_i], y[tr_i]), batch_size=BATCH_SIZE, shuffle=True, collate_fn=coll)
        va_dl = DataLoader(encode(tok, train.clean[va_i]), batch_size=64, collate_fn=coll)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        steps = EPOCHS * len(tr_dl)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')
        best_f1 = -1
        for ep in range(EPOCHS):
            model.train(); t0 = time.time()
            for b in tr_dl:
                b = {k: v.to(device) for k, v in b.items()}
                labels = b.pop('labels')
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=device == 'cuda'):
                    logits = model(**b).logits.float()
                    loss = torch.nn.functional.cross_entropy(logits[:, idx_t], labels, label_smoothing=0.05)
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step()
            p = predict(model, va_dl, idx)
            f1 = f1_score(y[va_i], p.argmax(1), average='macro')
            print(f'fold {f} epoch {ep}: val F1 {f1:.4f} ({time.time() - t0:.0f}s)')
            if f1 > best_f1:  # keep best epoch
                best_f1 = f1
                oof[va_i] = p
                fold_test = predict(model, test_dl, idx)
        test_prob += fold_test / len(folds)
        del model, opt
        torch.cuda.empty_cache()
    print(f'{name} OOF macro-F1: {f1_score(y, oof.argmax(1), average="macro"):.4f}')
    return oof, test_prob

In [ ]:
results = {}
# Reuse predictions saved by an earlier run (same folds) instead of retraining
for path in glob.glob('/kaggle/input/**/oof_*.npy', recursive=True):
    results[os.path.basename(path)[4:-4]] = (np.load(path), np.load(path.replace('oof_', 'test_')))
    print('loaded', os.path.basename(path))

for name, cfg in MODELS.items():
    tag = name.split('/')[-1]
    if tag in results:
        continue
    results[tag] = run_model(name, cfg['lr'], cfg['epochs'])
    np.save(f'oof_{tag}.npy', results[tag][0])
    np.save(f'test_{tag}.npy', results[tag][1])
print(list(results))

### 3.3 Model comparison, ensemble and per-class bias tuning

In [ ]:
def tune_bias(p, y, grid=np.linspace(-0.3, 0.3, 61), rounds=3):
    # Coordinate search of one additive offset per class on OOF probabilities
    b = np.zeros(p.shape[1])
    for _ in range(rounds):
        for c in range(p.shape[1]):
            s = [f1_score(y, (p + np.where(np.arange(len(b)) == c, g, b)).argmax(1), average='macro') for g in grid]
            b[c] = grid[int(np.argmax(s))]
    return b

runs = dict(results)
runs['ensemble'] = (np.mean([r[0] for r in results.values()], axis=0),   # average of probabilities
                    np.mean([r[1] for r in results.values()], axis=0))
final = {}
for tag, (oof, prob) in runs.items():
    bias = tune_bias(oof, y)
    f_arg = f1_score(y, oof.argmax(1), average='macro')
    f_tun = f1_score(y, (oof + bias).argmax(1), average='macro')
    print(f'{tag:40s} OOF F1 argmax {f_arg:.4f} | tuned {f_tun:.4f}')
    final[tag] = (oof, prob, bias, f_tun)

best_tag = max(final, key=lambda k: final[k][3])
oof, prob, bias, _ = final[best_tag]
print('Best:', best_tag)
print(classification_report(y, (oof + bias).argmax(1), target_names=LABELS, digits=3))

In [ ]:
for tag, (o, p, b, _) in final.items():
    save_submission((p + b).argmax(1), f'submission_{tag}.csv')

**Output of the two Kaggle runs (GPU T4, ~22 min each):**

```
run 1 - j-hartmann   best val F1 per fold: 0.6374 | 0.5873 | 0.6552 | 0.6646 | 0.6203   OOF 0.6334
run 2 - cardiffnlp   best val F1 per fold: 0.5880 | 0.6084 | 0.6273 | 0.6540 | 0.6050   OOF 0.6172
loaded emotion-english-roberta-large
twitter-roberta-large-emotion-latest     OOF F1 argmax 0.6172 | tuned 0.6284
emotion-english-roberta-large            OOF F1 argmax 0.6334 | tuned 0.6356
ensemble                                 OOF F1 argmax 0.6447 | tuned 0.6497
```
Public leaderboard: TF-IDF baseline 0.5378, j-hartmann 0.6347, ensemble 0.6346.